In [1]:
#undef __noinline__

# Introduction to CUDA

CUDA lets you run code directly on the GPU by writing special functions called **kernels**.
This notebook walks through the simplest possible examples to get you comfortable with the
three things every CUDA program has to do:

1. Move data onto the GPU
2. Launch a kernel to process it
3. Move the result back to the CPU

---

## Part 1 — Adding two numbers on the GPU

The `__global__` keyword tells the compiler that this function runs on the GPU but is called from the CPU.
The result can't be returned normally, so we write it through a pointer that lives in GPU memory.

In [2]:
__global__ void gpu_add(int a, int b, int *result) {
    *result = a + b;
}

To call it we use the `<<<blocks, threads>>>` launch syntax — `<<<1,1>>>` means one block,
one thread. Before the call we need a place in GPU memory to hold the answer, which we get
with `cudaMalloc`. After the kernel finishes we pull the value back with `cudaMemcpy`.

In [3]:
#include <cstdio>

int host_result;
int *dev_result;

cudaMalloc((void**)&dev_result, sizeof(int));

gpu_add<<<1, 1>>>(5, 9, dev_result);

cudaMemcpy(&host_result, dev_result, sizeof(int), cudaMemcpyDeviceToHost);

printf("5 + 9 = %d\n", host_result);
cudaFree(dev_result);

5 + 9 = 14


---

## Part 2 — Adding two arrays in parallel

A single GPU thread is no faster than the CPU. The power comes from launching **many threads at once**,
each one handling one element independently.

Below, the kernel adds a single pair of elements. The index it operates on comes from
`blockIdx.x` — the block number — so launching N blocks gives us N simultaneous additions.

![Vector_Add_Model](images/vectoradd.png)

In [4]:
#define N 10

__global__ void add_vectors(int *a, int *b, int *c) {
    int i = blockIdx.x;       // each block handles one element
    if (i < N)
        c[i] = a[i] + b[i];
}

We allocate three arrays on the GPU, copy the inputs across, launch `N` blocks, then bring
the result back.

The diagram below shows how blocks and threads are arranged inside the CUDA grid.

![CudaGrid](images/CudaGrid.png)

In [5]:
int h_in1[N], h_in2[N], h_out[N];
int *d_in1, *d_in2, *d_out;

cudaMalloc((void**)&d_in1, N * sizeof(int));
cudaMalloc((void**)&d_in2, N * sizeof(int));
cudaMalloc((void**)&d_out, N * sizeof(int));

for (int i = 0; i < N; i++) {
    h_in1[i] = i;
    h_in2[i] = i * i;
}

cudaMemcpy(d_in1, h_in1, N * sizeof(int), cudaMemcpyHostToDevice);
cudaMemcpy(d_in2, h_in2, N * sizeof(int), cudaMemcpyHostToDevice);

add_vectors<<<N, 1>>>(d_in1, d_in2, d_out);

cudaMemcpy(h_out, d_out, N * sizeof(int), cudaMemcpyDeviceToHost);

for (int i = 0; i < N; i++)
    printf("%d + %d = %d\n", h_in1[i], h_in2[i], h_out[i]);

cudaFree(d_in1);
cudaFree(d_in2);
cudaFree(d_out);

0 + 0 = 0
1 + 1 = 2
2 + 4 = 6
3 + 9 = 12
4 + 16 = 20
5 + 25 = 30
6 + 36 = 42
7 + 49 = 56
8 + 64 = 72
9 + 81 = 90
